## Feature Engineering - Dataset Boston Housing

### By:
Rafael Garcia

### Date:
2026-08-20

### Description:

Este notebook corresponde a la **Tarea 4: Feature Engineering** del Proyecto 1 (Dataset Estático) del curso de MLOps.

El objetivo es realizar el proceso de limpieza, transformación y preparación de los datos para entrenar un modelo de ML, utilizando transformadores y pipelines de scikit-learn. Se incluye:

1. Limpieza de datos (duplicados, outliers, valores faltantes)
2. Selección de atributos (Feature Selection)
3. Ingeniería de atributos (Feature Engineering)
4. Escalado de atributos (Feature Scaling)
5. Encoding de variables categóricas

## 📚 Import libraries

In [38]:
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

## 💾 Load data

In [39]:
df = pd.read_parquet("../../data/02_intermediate/boston_housing_clean.parquet")
print(f"Dimensiones originales: {df.shape}")
df.head()

Dimensiones originales: (448, 15)


,ID,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,black,lstat,medv
0,1,0.00632,18.0,2.31,0.0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98,24.0
1,2,0.02731,0.0,7.07,0.0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14,21.6
2,4,0.03237,0.0,2.18,0.0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94,33.4
3,5,0.06905,0.0,2.18,0.0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33,NaN
4,7,0.08829,12.5,7.87,0.0,0.524,6.012,66.6,5.5605,5,311.0,15.2,395.60,12.43,22.9


## 🧹 Paso 1: Limpieza de datos

### 1.1 Eliminar columna ID y filas sin variable objetivo

In [40]:
TARGET = "medv"
ID_COL = "ID"

# Eliminar columna ID (no aporta información predictiva)
df = df.drop(columns=[ID_COL])

# Eliminar filas donde la variable objetivo es nula
n_before = len(df)
df = df.dropna(subset=[TARGET])
n_after = len(df)
print(f"Filas eliminadas por medv nulo: {n_before - n_after}")
print(f"Dimensiones después de limpieza: {df.shape}")

Filas eliminadas por medv nulo: 16
Dimensiones después de limpieza: (432, 14)


### 1.2 Eliminar filas duplicadas

In [41]:
n_duplicados = df.duplicated().sum()
print(f"Filas duplicadas: {n_duplicados}")

if n_duplicados > 0:
    df = df.drop_duplicates()
    print(f"Dimensiones después de eliminar duplicados: {df.shape}")

Filas duplicadas: 89
Dimensiones después de eliminar duplicados: (343, 14)


### 1.3 Separar features y target

In [42]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

print(f"Features (X): {X.shape}")
print(f"Target (y): {y.shape}")
print(f"\nColumnas: {list(X.columns)}")
print("\nNulos en features:")
print(X.isnull().sum())

Features (X): (343, 13)
Target (y): (343,)

Columnas: ['crim', 'zn', 'indus', 'chas', 'nox', 'rm', 'age', 'dis', 'rad', 'tax', 'ptratio', 'black', 'lstat']

Nulos en features:
crim       2
zn         1
indus      5
chas       1
nox        3
rm         4
age        3
dis        4
rad        3
tax        3
ptratio    2
black      3
lstat      1
dtype: int64


### 1.4 División Train / Test

Se divide antes del feature engineering para evitar data leakage.

In [43]:
TEST_SIZE = 0.2
RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape[0]} filas ({X_train.shape[0] / len(X) * 100:.0f}%)")
print(f"Test:  {X_test.shape[0]} filas ({X_test.shape[0] / len(X) * 100:.0f}%)")

Train: 274 filas (80%)
Test:  69 filas (20%)


## 🔧 Paso 2: Definición de grupos de columnas

Se definen los grupos de columnas según el tipo de transformación que requieren, con base en los hallazgos del EDA.

In [44]:
# Columnas que requieren transformación logarítmica (alta asimetría detectada en EDA)
LOG_COLS = ["crim", "zn", "dis", "lstat"]

# Columnas numéricas sin transformación log
NUM_COLS = ["indus", "nox", "rm", "age", "tax", "ptratio", "black"]

# Columnas categóricas
CAT_COLS = ["chas"]

# Columnas ordinales (tratadas como numéricas)
ORD_COLS = ["rad"]

print(f"Log transform: {LOG_COLS}")
print(f"Numéricas:     {NUM_COLS}")
print(f"Categóricas:   {CAT_COLS}")
print(f"Ordinales:     {ORD_COLS}")
print(f"\nTotal columnas: {len(LOG_COLS) + len(NUM_COLS) + len(CAT_COLS) + len(ORD_COLS)}")
print(f"Columnas en X:  {X.shape[1]}")

Log transform: ['crim', 'zn', 'dis', 'lstat']
Numéricas:     ['indus', 'nox', 'rm', 'age', 'tax', 'ptratio', 'black']
Categóricas:   ['chas']
Ordinales:     ['rad']

Total columnas: 13
Columnas en X:  13


## ⚙️ Paso 3: Construcción de pipelines de scikit-learn

Se construyen pipelines separados para cada tipo de variable y se combinan con un `ColumnTransformer`.

### 3.1 Transformador personalizado para log(1+x)

In [45]:
# Transformador log(1+x) usando FunctionTransformer
log_transformer = FunctionTransformer(np.log1p, validate=True)

### 3.2 Pipeline para variables numéricas con transformación log

1. Imputar nulos con la mediana
2. Aplicar log(1+x)
3. Escalar con StandardScaler

In [46]:
log_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        (
            "log_transform",
            FunctionTransformer(np.log1p, validate=True, feature_names_out="one-to-one"),
        ),
        ("scaler", StandardScaler()),
    ]
)

print("Pipeline log:")
print(log_pipeline)

Pipeline log:
Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('log_transform',
                 FunctionTransformer(feature_names_out='one-to-one',
                                     func=<ufunc 'log1p'>, validate=True)),
                ('scaler', StandardScaler())])


### 3.3 Pipeline para variables numéricas estándar

1. Imputar nulos con la mediana
2. Escalar con StandardScaler

In [47]:
num_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

print("Pipeline numérico:")
print(num_pipeline)

Pipeline numérico:
Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])


### 3.4 Pipeline para variables categóricas

1. Imputar nulos con el valor más frecuente
2. Aplicar OneHotEncoder

In [48]:
cat_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(drop="if_binary", sparse_output=False, handle_unknown="ignore")),
    ]
)

print("Pipeline categórico:")
print(cat_pipeline)

Pipeline categórico:
Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder',
                 OneHotEncoder(drop='if_binary', handle_unknown='ignore',
                               sparse_output=False))])


### 3.5 Pipeline para variables ordinales

1. Imputar nulos con la mediana
2. Escalar con StandardScaler (se trata como numérica ordinal)

In [49]:
ord_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

print("Pipeline ordinal:")
print(ord_pipeline)

Pipeline ordinal:
Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])


### 3.6 ColumnTransformer: combinar todos los pipelines

In [50]:
preprocessor = ColumnTransformer(
    transformers=[
        ("log", log_pipeline, LOG_COLS),
        ("num", num_pipeline, NUM_COLS),
        ("cat", cat_pipeline, CAT_COLS),
        ("ord", ord_pipeline, ORD_COLS),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)

print("Preprocessor completo:")
print(preprocessor)

Preprocessor completo:
ColumnTransformer(transformers=[('log',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('log_transform',
                                                  FunctionTransformer(feature_names_out='one-to-one',
                                                                      func=<ufunc 'log1p'>,
                                                                      validate=True)),
                                                 ('scaler', StandardScaler())]),
                                 ['crim', 'zn', 'dis', 'lstat']),
                                ('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                             

## 🚀 Paso 4: Ajustar y transformar los datos

In [51]:
# Convertir chas a string para que OneHotEncoder funcione correctamente
X_train["chas"] = X_train["chas"].astype(str)
X_test["chas"] = X_test["chas"].astype(str)

# Convertir rad a float para SimpleImputer
X_train["rad"] = X_train["rad"].astype(float)
X_test["rad"] = X_test["rad"].astype(float)

# Fit en train, transform en ambos
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"X_train procesado: {X_train_processed.shape}")
print(f"X_test procesado:  {X_test_processed.shape}")

X_train procesado: (274, 13)
X_test procesado:  (69, 13)


In [52]:
# Obtener nombres de las features procesadas
feature_names = preprocessor.get_feature_names_out()
print(f"\nFeatures procesadas ({len(feature_names)}):")
for i, name in enumerate(feature_names):
    print(f"  {i + 1}. {name}")


Features procesadas (13):
  1. log__crim
  2. log__zn
  3. log__dis
  4. log__lstat
  5. num__indus
  6. num__nox
  7. num__rm
  8. num__age
  9. num__tax
  10. num__ptratio
  11. num__black
  12. cat__chas_1.0
  13. ord__rad


In [53]:
# Verificar que no hay nulos después del procesamiento
train_nulls = np.isnan(X_train_processed).sum()
test_nulls = np.isnan(X_test_processed).sum()
print(f"Nulos en train procesado: {train_nulls}")
print(f"Nulos en test procesado:  {test_nulls}")

Nulos en train procesado: 0
Nulos en test procesado:  0


In [54]:
# Crear DataFrames con los datos procesados para inspección
df_train_processed = pd.DataFrame(X_train_processed, columns=feature_names)
print("=== Estadísticas del dataset procesado (train) ===")
df_train_processed.describe().round(4)

=== Estadísticas del dataset procesado (train) ===


,log__crim,log__zn,log__dis,log__lstat,num__indus,num__nox,num__rm,num__age,num__tax,num__ptratio,num__black,cat__chas_1.0,ord__rad
count,274.0000,274.0000,274.0000,274.0000,274.0000,274.0000,274.0000,274.0000,274.0000,274.0000,274.0000,274.0000,274.0000
mean,-0.0000,0.0000,-0.0000,0.0000,-0.0000,0.0000,0.0000,-0.0000,0.0000,0.0000,-0.0000,0.0474,0.0000
std,1.0018,1.0018,1.0018,1.0018,1.0018,1.0018,1.0018,1.0018,1.0018,1.0018,1.0018,0.2130,1.0018
min,-0.7925,-0.5923,-1.8219,-2.7711,-1.4990,-1.4439,-3.5583,-2.2003,-1.3083,-2.7784,-3.9956,0.0000,-0.9816
25%,-0.7221,-0.5923,-0.8524,-0.6941,-0.8668,-0.9014,-0.5628,-0.8237,-0.7623,-0.5208,0.2123,0.0000,-0.6359
50%,-0.5615,-0.5923,-0.1586,0.0545,-0.2102,-0.1810,-0.1222,0.3592,-0.4746,0.2553,0.3895,0.0000,-0.5207
75%,0.6527,1.1713,0.8746,0.7642,1.0009,0.7090,0.4716,0.8891,1.4982,0.7961,0.4341,0.0000,1.6688
max,3.5045,2.2745,2.4869,2.1843,2.3891,2.6415,3.6365,1.1091,1.7624,1.2665,0.4412,1.0000,1.6688


## 📋 Paso 5: Resumen de transformaciones aplicadas

In [55]:
print("=" * 65)
print("RESUMEN DE TRANSFORMACIONES APLICADAS")
print("=" * 65)
print()
print("1. LIMPIEZA:")
print("   - Eliminada columna ID (no predictiva)")
print("   - Eliminadas filas con medv nulo (variable objetivo)")
print("   - Verificados y eliminados duplicados")
print()
print("2. IMPUTACIÓN DE NULOS:")
print("   - Variables numéricas y ordinales: mediana")
print("   - Variables categóricas: valor más frecuente")
print()
print("3. FEATURE ENGINEERING:")
print(f"   - Transformación log(1+x): {LOG_COLS}")
print("   - Reduce asimetría en variables con distribución sesgada")
print()
print("4. FEATURE SCALING:")
print("   - StandardScaler (media=0, std=1) en todas las numéricas")
print()
print("5. ENCODING:")
print("   - OneHotEncoder (drop='if_binary') para chas")
print("   - rad tratada como numérica ordinal (escalada)")
print()
print(f"Dimensiones finales: {X_train_processed.shape[1]} features")
print(f"Train: {X_train_processed.shape[0]} filas")
print(f"Test:  {X_test_processed.shape[0]} filas")

RESUMEN DE TRANSFORMACIONES APLICADAS

1. LIMPIEZA:
   - Eliminada columna ID (no predictiva)
   - Eliminadas filas con medv nulo (variable objetivo)
   - Verificados y eliminados duplicados

2. IMPUTACIÓN DE NULOS:
   - Variables numéricas y ordinales: mediana
   - Variables categóricas: valor más frecuente

3. FEATURE ENGINEERING:
   - Transformación log(1+x): ['crim', 'zn', 'dis', 'lstat']
   - Reduce asimetría en variables con distribución sesgada

4. FEATURE SCALING:
   - StandardScaler (media=0, std=1) en todas las numéricas

5. ENCODING:
   - OneHotEncoder (drop='if_binary') para chas
   - rad tratada como numérica ordinal (escalada)

Dimensiones finales: 13 features
Train: 274 filas
Test:  69 filas


## 💾 Paso 6: Guardar el pipeline y los datos procesados

In [56]:
# Guardar el preprocessor entrenado
pipeline_path = "../../models/preprocessor.joblib"
joblib.dump(preprocessor, pipeline_path)
print(f"Pipeline guardado en: {pipeline_path}")

# Verificar carga
preprocessor_loaded = joblib.load(pipeline_path)
X_test_verify = preprocessor_loaded.transform(X_test)
print(f"Verificación: shapes coinciden = {X_test_verify.shape == X_test_processed.shape}")
print(
    f"Verificación: valores iguales = {np.allclose(X_test_verify, X_test_processed, equal_nan=True)}"
)

Pipeline guardado en: ../../models/preprocessor.joblib
Verificación: shapes coinciden = True
Verificación: valores iguales = True


In [57]:
# Guardar datos procesados para la siguiente tarea
train_path = "../../data/05_model_input/X_train.parquet"
test_path = "../../data/05_model_input/X_test.parquet"
y_train_path = "../../data/05_model_input/y_train.parquet"
y_test_path = "../../data/05_model_input/y_test.parquet"

pd.DataFrame(X_train_processed, columns=feature_names).to_parquet(train_path, index=False)
pd.DataFrame(X_test_processed, columns=feature_names).to_parquet(test_path, index=False)
y_train.reset_index(drop=True).to_frame().to_parquet(y_train_path, index=False)
y_test.reset_index(drop=True).to_frame().to_parquet(y_test_path, index=False)

print("Datos guardados en data/05_model_input/")
print(f"  X_train: {train_path}")
print(f"  X_test:  {test_path}")
print(f"  y_train: {y_train_path}")
print(f"  y_test:  {y_test_path}")

Datos guardados en data/05_model_input/
  X_train: ../../data/05_model_input/X_train.parquet
  X_test:  ../../data/05_model_input/X_test.parquet
  y_train: ../../data/05_model_input/y_train.parquet
  y_test:  ../../data/05_model_input/y_test.parquet


## 📊 Analysis of Results and Conclusions

### Resultados

Se construyó un pipeline de preprocesamiento completo usando `ColumnTransformer` de scikit-learn que integra 4 sub-pipelines:

| Pipeline | Columnas | Transformaciones |
|----------|----------|------------------|
| `log` | crim, zn, dis, lstat | Imputación (mediana) → log(1+x) → StandardScaler |
| `num` | indus, nox, rm, age, tax, ptratio, black | Imputación (mediana) → StandardScaler |
| `cat` | chas | Imputación (moda) → OneHotEncoder |
| `ord` | rad | Imputación (mediana) → StandardScaler |

### Conclusiones

- El pipeline procesa 13 features de entrada y produce 13 features de salida (chas binaria se mantiene como 1 columna con `drop='if_binary'`).
- Todos los valores nulos se eliminaron mediante imputación.
- Las variables con alta asimetría se transformaron con log(1+x) para acercarlas a distribuciones normales.
- El pipeline se ajusta solo con datos de entrenamiento (fit_transform en train, transform en test) para evitar data leakage.
- El preprocessor se serializó con joblib para poder reutilizarlo en las tareas siguientes.

## 💡 Proposals and Ideas

1. **Tarea 5 (Baseline)**: usar este pipeline con un `DummyRegressor` para establecer el rendimiento mínimo.
2. **Tarea 6 (Selección de modelo)**: crear un `Pipeline` que una el preprocessor con diferentes modelos (`Pipeline([('preprocessor', preprocessor), ('model', model)])`).
3. **Mejoras potenciales al pipeline**:
   - Agregar `PolynomialFeatures` para capturar interacciones (rm², rm×lstat).
   - Probar `RobustScaler` en lugar de `StandardScaler` para mayor robustez a outliers.
   - Evaluar eliminación de `black` por consideraciones éticas.
   - Probar `KNNImputer` en lugar de `SimpleImputer` para imputación más sofisticada.

## 📖 References

- Guía del proyecto (Tarea 4): [joserzapata.github.io/post/ciencia-datos-proyecto-python/4-feat_eng/](https://joserzapata.github.io/post/ciencia-datos-proyecto-python/4-feat_eng/)
- Scikit-learn Preprocessing: [sklearn.preprocessing](https://scikit-learn.org/stable/modules/preprocessing.html)
- Scikit-learn Pipelines: [sklearn.compose](https://scikit-learn.org/stable/modules/compose.html)
- Scikit-learn ColumnTransformer: [sklearn.compose.ColumnTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html)